In [45]:
import hashlib
import os
import random
import time
import urllib.request
import zipfile

import pandas as pd

In [46]:
class Shingle():
    
    def __init__(self,k,doc):   
        self.k = k  #number of letters of each shingle
        self.doc = doc
        self.shingles = set()   
        words = " ".join(word.lower() for word in doc.split()) 
        if len(words) < k:
            # document shorter than k (e.g. the SMS "Ok"): use the whole text as its only shingle,
            # otherwise the set would be empty and minhashing would fail (min of an empty list)
            self.shingles.add(words)
            return
        noofshingles = (len(words) - k) +1 #stop at last possible shingle letter (number of shingles in the text)
        for i in range(noofshingles): 
            shingle = words[i:i+k]
            self.shingles.add(shingle)  #its a set so there are no duplicates

    def hashes(self):
        s = sorted(int.from_bytes(hashlib.sha256(shingle.encode("utf-8")).digest()[:4]) for shingle in self.shingles)  #converts each shingle in a number
        return set(s)

In [47]:
class CompareSets():
    def sim(self,a,b):
        return len(a.intersection(b)) / len(a.union(b))   #number of elements in common divided by total number of elements 

In [48]:
class Minhashing():
    # Builds a minhash signature (a vector of n integers) from a set of hashed shingles.
    # Idea: instead of really permuting the rows of the (shingles x documents) matrix,
    # each random hash function h(x) = (a*x + b) % p simulates one permutation:
    # h(x) is the "new position" of shingle x after shuffling the rows.
    def __init__(self, n, seed=42):
        # n = length of the signature = number of simulated permutations (hash functions).
        self.rng = random.Random(seed)   #for reproducibility (always the same functions)
        self.hashFunctions = self.createHashFunctions(n)

    def signature(self, hashSet):
        # hashSet = set of hashed shingles of one document (output of Shingle.hashes()).
        sig = []
        for h in self.hashFunctions:                  # one iteration per permutation
            values = [h(x) for x in hashSet]          # new position of every shingle the document has
            sig.append(min(values))                   # smallest position = "first row with a 1" in the permuted matrix
        return sig                                    # sig[i] = minhash value of the document for permutation i

    def createHashFunctions(self, n):
        # Creates n random hash functions of the form h(x) = (a*x + b) % p.
        funcs = []
        p = 4294967311  # smallest prime above 2^32: bigger than any shingle hash (0 .. 2^32-1)
        for _ in range(n):
            a = self.rng.randint(1, p - 1)
            b = self.rng.randint(0, p - 1)
            f = lambda x, a=a, b=b: (a * x + b) % p  #function with different a and different b every time
            funcs.append(f)
        return funcs


In [49]:
class CompareSignatures():
    # Estimates the Jaccard similarity of two documents from their minhash signatures:
    # the fraction of positions in which the two signatures have the same value.
    # Both signatures must come from the SAME Minhashing object (same hash functions, same order).
    def sim(self, sig1, sig2):
        if len(sig1) != len(sig2):
            raise ValueError("Signatures must have the same length")
        agree = 0
        for x, y in zip(sig1, sig2):   # zip walks both lists at the same time: (sig1[0], sig2[0]), (sig1[1], sig2[1]), ...
            if x == y:
                agree += 1
        return agree / len(sig1)

In [50]:
a={1,2,3}
b={1,2,3}
c = CompareSets()
c.sim(a,b)
a = Shingle(3, "aple").hashes()
b= Shingle(3, "apple").hashes()
c = CompareSets()
c.sim(a,b)

m = Minhashing(100)
sa = m.signature(Shingle(3, "apple").hashes())
sb = m.signature(Shingle(3, "aple").hashes())
print("min hash set",sa)
print("min hash set",sb)
len(sa)    


m2= Minhashing(1000)   
sa2 = m2.signature(Shingle(3, "apple").hashes())
sb2 = m2.signature(Shingle(3, "aple").hashes())

m3= Minhashing(10000)   
sa3 = m3.signature(Shingle(3, "apple").hashes())
sb3 = m3.signature(Shingle(3, "aple").hashes())

# Compare the real Jaccard similarity with the estimate from the minhash signatures
print("Jaccard (real):            ", CompareSets().sim(Shingle(3, "apple").hashes(), Shingle(3, "aple").hashes()))
print("Signatures 100 (estimate):     ", CompareSignatures().sim(sa, sb))
print("Signatures 1000 (estimate):     ", CompareSignatures().sim(sa2, sb2))
print("Signatures 10000 (estimate):     ", CompareSignatures().sim(sa3, sb3))



min hash set [1017222307, 1682720542, 87706185, 443545964, 297043260, 46945462, 440769173, 167518471, 1116642240, 2093057053, 161682048, 656563295, 973878717, 2604929466, 377759614, 1368757743, 12406329, 238106786, 381219020, 239098944, 3170230382, 886530584, 575230851, 458713562, 353060383, 1009678050, 629993637, 740668653, 1542361980, 240918586, 1792615796, 1422941906, 2766932556, 755281232, 1084611792, 666235202, 191173765, 1267373707, 810846926, 1628731697, 1009905189, 895773080, 2856890991, 2624740740, 108214023, 986747687, 1781136100, 1932250351, 1035510953, 913429487, 691203967, 2185893827, 197958304, 1319529795, 1325727064, 758842879, 502874174, 146370741, 405802080, 1250071512, 1994992472, 911885978, 360534798, 460518796, 1170463254, 699448365, 1594936929, 594551339, 613771070, 851438528, 367023494, 789066848, 1026017203, 262680327, 186084345, 945548414, 1868356778, 647170593, 431763467, 1697119407, 1162665358, 2218820684, 2268112420, 1333380835, 135792804, 145244288, 81063388

## Dataset

We use the **SMS Spam Collection** from the UC Irvine Machine Learning Repository:
https://archive.ics.uci.edu/dataset/228/sms+spam+collection

It contains 5,574 real SMS messages (each line: `label<TAB>message`). Each message is one document.
Spam messages are often sent many times with small changes, so the corpus contains real near-duplicates.

The dataset is **not** included in the submission: the cell below downloads it automatically into `data/` the first time it runs.

In [51]:
DATA_URL = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
DATA_DIR = "data"

def load_sms_dataset():
    # Downloads the zip only if it is not already on disk, then returns the list of messages (documents).
    zip_path = os.path.join(DATA_DIR, "sms_spam_collection.zip")
    if not os.path.exists(zip_path):
        os.makedirs(DATA_DIR, exist_ok=True)
        print("Downloading dataset...")
        urllib.request.urlretrieve(DATA_URL, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        text = z.read("SMSSpamCollection").decode("utf-8")
    docs = []
    for line in text.splitlines():
        label, message = line.split("\t", 1)   # split only on the first tab: "ham" / "spam" and the message
        docs.append(message)
    return docs

docs = load_sms_dataset()
print(len(docs), "documents")
docs[:3]

5574 documents


['Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...',
 'Ok lar... Joking wif u oni...',
 "Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's"]

## Finding similar documents

Steps:
1. Get the shingles of each document and hash them
2. Make the minhash signature of each document
3. Compare every pair of documents with Jaccard (real similarity)
4. Compare every pair of signatures (estimated similarity)

Two documents are similar if the similarity is >= s.

We use k = 5 because the messages are short (lecture 2 says k = 5 is ok for short documents like emails), n = 100 and s = 0.8.

In [ ]:
def find_similar_documents(docs, k=5, n=100, s=0.8):
    times = {}

    # 1. Shingling: one set of hashed shingles per document
    start = time.perf_counter()
    shingle_sets = [Shingle(k, doc).hashes() for doc in docs]
    times["shingling (s)"] = time.perf_counter() - start

    # 2. Minhashing: one signature of length n per document
    start = time.perf_counter()
    minhashing = Minhashing(n)
    signatures = [minhashing.signature(hash_set) for hash_set in shingle_sets]
    times["minhashing (s)"] = time.perf_counter() - start

    compare_sets = CompareSets()
    compare_signatures = CompareSignatures()

    # 3. Exact Jaccard similarity of every pair (i, j) with i < j, so each pair is checked once
    start = time.perf_counter()
    jaccard_pairs = {}
    for i in range(len(docs)):
        for j in range(i + 1, len(docs)):
            sim = compare_sets.sim(shingle_sets[i], shingle_sets[j])
            if sim >= s:
                jaccard_pairs[(i, j)] = sim
    times["compare sets (s)"] = time.perf_counter() - start

    # 4. Estimated similarity of every pair using the minhash signatures
    start = time.perf_counter()
    signature_pairs = {}
    for i in range(len(docs)):
        for j in range(i + 1, len(docs)):
            sim = compare_signatures.sim(signatures[i], signatures[j])
            if sim >= s:
                signature_pairs[(i, j)] = sim
    times["compare signatures (s)"] = time.perf_counter() - start

    return times, jaccard_pairs, signature_pairs




In [ ]:
# run everything with 10, 100 and 1000 documents and measure how long each step takes
K, N_HASHES, S = 5, 100, 0.8
results = []
for size in [10, 100, 1000]:
    times, jaccard_pairs, signature_pairs = find_similar_documents(docs[:size], k=K, n=N_HASHES, s=S)
    both = set(jaccard_pairs) & set(signature_pairs)
    results.append({
        "documents": size,
        **times,
        "total (s)": sum(times.values()),
        "similar pairs (Jaccard)": len(jaccard_pairs),
        "similar pairs (signatures)": len(signature_pairs),
        "found by both": len(both),
        "false positives": len(set(signature_pairs) - set(jaccard_pairs)),  # signatures say similar, Jaccard says no (jaccard was less than 0.8 and signatures was more)
        "false negatives": len(set(jaccard_pairs) - set(signature_pairs)),  # Jaccard says similar, signatures missed it
    })

results_df = pd.DataFrame(results).round(3)
results_df

,documents,shingling (s),minhashing (s),compare sets (s),compare signatures (s),total (s),similar pairs (Jaccard),similar pairs (signatures),found by both,false positives,false negatives
0,10,0.001,0.018,0.000,0.000,0.019,0,0,0,0,0
1,100,0.007,0.132,0.019,0.011,0.170,0,0,0,0,0
2,1000,0.075,1.334,2.120,1.215,4.744,40,40,39,1,1


In [ ]:
# similar pairs found with 1000 documents, with the real and the estimated similarity
rows = []
for (i, j) in sorted(set(jaccard_pairs) | set(signature_pairs)):
    rows.append({
        "doc i": i,
        "doc j": j,
        "Jaccard": round(CompareSets().sim(Shingle(K, docs[i]).hashes(), Shingle(K, docs[j]).hashes()), 3),  #here we could have written  "<s" for false positives but we are going to calculate it instead
        "signatures": signature_pairs.get((i, j), "< s"),  #here we choose not to recalculate it
        "text i": docs[i],
        "text j": docs[j],
    })
pd.set_option("display.max_colwidth", 100)
pd.DataFrame(rows).sort_values("Jaccard").head(15)

,doc i,doc j,Jaccard,signatures,text i,text j
16,135,474,0.760,0.8,Want 2 get laid tonight? Want real Dogging locations sent direct 2 ur mob? Join the UK's largest...,Want 2 get laid tonight? Want real Dogging locations sent direct 2 ur Mob? Join the UK's largest...
40,879,880,0.811,0.85,U have a secret admirer who is looking 2 make contact with U-find out who they R*reveal who thin...,U have a Secret Admirer who is looking 2 make contact with U-find out who they R*reveal who thin...
15,117,160,0.823,0.88,You are a winner U have been specially selected 2 receive £1000 or a 4* holiday (flights inc) sp...,You are a winner U have been specially selected 2 receive £1000 cash or a 4* holiday (flights in...
4,76,183,0.829,< s,Thats cool. i am a gentleman and will treat you with dignity and respect.,ok. I am a gentleman and will treat you with dignity and respect.
23,240,879,0.864,0.83,U 447801259231 have a secret admirer who is looking 2 make contact with U-find out who they R*re...,U have a secret admirer who is looking 2 make contact with U-find out who they R*reveal who thin...
32,389,710,0.911,0.86,4mths half price Orange line rental & latest camera phones 4 FREE. Had your phone 11mths ? Call ...,4mths half price Orange line rental & latest camera phones 4 FREE. Had your phone 11mths+? Call ...
3,56,492,1.000,1.0,"Congrats! 1 year special cinema pass for 2 is yours. call 09061209465 now! C Suprman V, Matrix3,...","Congrats! 1 year special cinema pass for 2 is yours. call 09061209465 now! C Suprman V, Matrix3,..."
5,80,223,1.000,1.0,"Sorry, I'll call later","Sorry, I'll call later"
7,80,444,1.000,1.0,"Sorry, I'll call later","Sorry, I'll call later"
8,80,702,1.000,1.0,"Sorry, I'll call later","Sorry, I'll call later"
